# S02 · Lab 1 — From witnesses to an AMIE output decision

## [Submit Lab 1 on Nextcloud](https://nextcloud.lisn.upsaclay.fr/index.php/s/WXyy8xrgKzjExAd)

Use the same password as for the Machine Learning labs. Enter the instructor’s surname in **uppercase** followed by **2026**, with no spaces.

Upload your completed notebook with your written answers.

Open the notebook with the course Python kernel and run the cells in order. Keep `rule_engine.py` and `data/` in the lab folder above `student/`.

The synthetic graph in `toy.tsv` uses three relations: `interacts_with(gene,compound)`, `treats(compound,disease)` and `associated_with(gene,disease)`.

In [1]:
from pathlib import Path
import sys
options = [Path.cwd(), Path.cwd() / "materials" / "rule-mining", *Path.cwd().parents]
BASE = next(p for p in options if (p / "rule_engine.py").is_file())
sys.path.insert(0, str(BASE))
from rule_engine import Atom, Rule, read_triples, relation_index, match_body, score_rule

toy = relation_index(read_triples(BASE / "data/toy.tsv"))
rule = Rule(Atom("associated_with", "?g", "?d"),
            (Atom("interacts_with", "?g", "?m"), Atom("treats", "?m", "?d")))
print("Rule:", rule.label())
for name, pairs in sorted(toy.items()):
    print(name, sorted(pairs))

Rule: interacts_with(?g,?m) & treats(?m,?d) => associated_with(?g,?d)
associated_with [('g0', 'd0'), ('g1', 'd1'), ('g2', 'd2'), ('g3', 'd3'), ('g4', 'd9')]
interacts_with [('g1', 'm1'), ('g1', 'm1bis'), ('g2', 'm2'), ('g3', 'm3'), ('g4', 'm4'), ('g5', 'm5'), ('g6', 'm6')]
treats [('m1', 'd1'), ('m1bis', 'd1'), ('m2', 'd2'), ('m3', 'd3'), ('m4', 'd4'), ('m5', 'd5'), ('m6', 'd6')]


## 1. Project the matches onto the head

Each body match is a dictionary that binds `?g`, `?m` and `?d` to identifiers. Complete `student_projection` so it returns a set of distinct `(gene, disease)` pairs. The join that produces the dictionaries is provided.

Before running the cell, predict whether the numbers of matches and distinct predictions will be equal. Explain.

**Answer** : I predict that it will be different, because there are a limited list of predictions deduced by the rule engine. These deductions may have multiple witnesses. Here, based on the data above, g1 interacts with m1 and m1bis, both treating d1. So for the (g1, d1) pair we get two body witnesses but only one distinct prediction.

In [3]:
def student_projection(bindings):
    return {(b["?g"], b["?d"]) for b in bindings}

bindings = match_body(rule.body, toy)
predictions = student_projection(bindings)
print("Body witnesses:", len(bindings))
print("Distinct predictions:", sorted(predictions))

Body witnesses: 7
Distinct predictions: [('g1', 'd1'), ('g2', 'd2'), ('g3', 'd3'), ('g4', 'd4'), ('g5', 'd5'), ('g6', 'd6')]


## 2. Add a second witness

In the original graph, list the predicted pairs that are recorded in `associated_with`. Then list the predictions counted in the **subject-oriented PCA denominator**: a predicted pair counts if its gene has any recorded `associated_with` value.

The cell below adds `interacts_with(g2,m2bis)` and `treats(m2bis,d2)` to a copy of the original graph. Before running it, predict what will happen to the number of body matches, the number of distinct predictions, support, standard confidence and PCA confidence. Check your predictions against the output.

**Answer** : What's already recorded is (g1, d1), (g2, d2), (g3, d3) ie: support = 3. PCA denominator is everything that's already recorded (even if it's not the same value predicted) so {(g1, d1), ..., (g4, d4)} and its value is thus 4. PCA confidence is support/PCA denom = 3/4. Standard confidence is support/total predictions = 3/6 = 1/2. Before running the cell, I predict that adding this won't change the number of distinct predictions, but it'll add a witness to (g2, d2) so body match will increase to 8. Support won't change, so both confidence too, i.e. still 3/4 and 1/2.

In [6]:
head = toy["associated_with"]
documented_genes = {gene for gene, _ in head}
positive = predictions & head
pca = {pair for pair in predictions if pair[0] in documented_genes}
initial_scores = score_rule(rule, toy)
print("Recorded intersection:", sorted(positive))
print("PCA denominator:", sorted(pca))
print("Original graph:", len(bindings), "witnesses", initial_scores.as_dict())

changed = {relation: set(pairs) for relation, pairs in toy.items()}
changed["interacts_with"].add(("g2", "m2bis"))
changed["treats"].add(("m2bis", "d2"))
changed_bindings = match_body(rule.body, changed)
changed_pairs = student_projection(changed_bindings)
print("Changed graph:", len(changed_bindings), "witnesses", score_rule(rule, changed).as_dict())
print("New projected pairs:", sorted(changed_pairs - predictions))

Recorded intersection: [('g1', 'd1'), ('g2', 'd2'), ('g3', 'd3')]
PCA denominator: [('g1', 'd1'), ('g2', 'd2'), ('g3', 'd3'), ('g4', 'd4')]
Original graph: 7 witnesses {'support': 3, 'body_pairs': 6, 'pca_pairs': 4, 'head_pairs': 5, 'confidence': 0.5, 'pca_confidence': 0.75, 'head_coverage': 0.6}
Changed graph: 8 witnesses {'support': 3, 'body_pairs': 6, 'pca_pairs': 4, 'head_pairs': 5, 'confidence': 0.5, 'pca_confidence': 0.75, 'head_coverage': 0.6}
New projected pairs: []


### Build a variant

Start again from the original graph. Choose a gene from `interacts_with` and a disease from `treats`. Add one `interacts_with` fact and one `treats` fact using the same new compound `m_new`. Keep all original facts.

Choose the two facts so that **standard confidence decreases and subject-oriented PCA confidence stays the same**. Before running the cell, write down the two facts, the pair they will add to the predictions, and the two expected confidence fractions. Explain your choice, then check it against the output.

**Answer** : I need a pair st, it's not supported (would not change #support), is not in the predicted set, so adding it would increase the number of distinct predictions by one and does not change PCA denom (ie: the gene is not in "associated with"). (g5, d6) is a good candidate for that

In [8]:
gene = "g5"     # Choose an existing gene identifier.
disease = "d6"  # Choose an existing disease identifier.
if not gene or not disease:
    raise ValueError("Choose a gene and a disease before running this cell.")

variant = {relation: set(pairs) for relation, pairs in toy.items()}
variant["interacts_with"].add((gene, "m_new"))
variant["treats"].add(("m_new", disease))

variant_pairs = student_projection(match_body(rule.body, variant))
print("New projected pairs:", sorted(variant_pairs - predictions))
print("Original graph:", initial_scores.as_dict())
print("Your variant:", score_rule(rule, variant).as_dict())

New projected pairs: [('g5', 'd6')]
Original graph: {'support': 3, 'body_pairs': 6, 'pca_pairs': 4, 'head_pairs': 5, 'confidence': 0.5, 'pca_confidence': 0.75, 'head_coverage': 0.6}
Your variant: {'support': 3, 'body_pairs': 7, 'pca_pairs': 4, 'head_pairs': 5, 'confidence': 0.42857142857142855, 'pca_confidence': 0.75, 'head_coverage': 0.6}


## 3. Read the AMIE output

Both runs use the original `toy.tsv`, target `associated_with`, and allow up to three atoms including the head. The cell below uses minimum support 1. Find the path rule from §1. Match **Support**, **Body Size** and **Pca Body Size** to the three sets you computed for the original graph. Verify both confidence values by division.

The default mode reads saved AMIE 4.0.0 output. If Java and the AMIE JAR are available, you can set `RUN_NEW = True` to run AMIE locally. Use the same mode at both thresholds.

In [12]:
import subprocess
import time

RUN_NEW = False

def inspect_amie(threshold):
    print("Minimum support:", threshold)
    command = ["java", "-Xmx1G", "-jar", str(BASE / "tools/amie4.0.0.jar"),
               "-bias", "default", "-enstd", "-noHeuristics", "-nc", "1",
               "-pm", "support", "-mins", str(threshold), "-minis", str(threshold),
               "-htr", "associated_with", "-maxad", "3", "-minpca", "0", "-minc", "0",
               str(BASE / "data/toy.tsv")]
    if RUN_NEW:
        start = time.monotonic()
        run = subprocess.run(command, cwd=BASE, capture_output=True, text=True,
                             check=True, timeout=30)
        output = run.stdout
        print("NEW RUN", command, "seconds", time.monotonic() - start)
    else:
        output = (BASE / f"data/runs/amie-toy-support{threshold}.txt").read_text()
        print("REPLAY: no new runtime measurement")
    lines = output.splitlines()
    columns = next(line.split("\t") for line in lines if line.startswith("Rule\t"))
    rows = [line.split("\t") for line in lines if "=>" in line]
    print("Native columns:", " | ".join(columns))
    print("Native rule rows:", len(rows))
    for row in rows:
        for field, value in zip(columns, row, strict=True):
            print(f"  {field}: {value}")

inspect_amie(1)

Minimum support: 1
REPLAY: no new runtime measurement
Native columns: Rule | Head Coverage | Standard Confidence | Pca Confidence | Support | Body Size | Pca Body Size | Functional Variable
Native rule rows: 1
  Rule: ?a  interacts_with  ?f  ?f  treats  ?b   => ?a  associated_with  ?b
  Head Coverage: 0.600000
  Standard Confidence: 0.500000
  Pca Confidence: 0.750000
  Support: 3
  Body Size: 6
  Pca Body Size: 4
  Functional Variable: -1


**Answer** : The path rule is this : ?a  interacts_with  ?f  ?f  treats  ?b   => ?a  associated_with  ?b ; The only thing are variables named differently. Support = $\#\{(g1, d1), (g2, d2), (g3, d3)\}$, body size = $\#\{(g1, d1), (g2, d2), (g3, d3), (g4, d4), (g5, d5), (g6, d6)\}$, PCA body size = $\#\{(g1, d1), (g2, d2), (g3, d3), (g4, d4)\}$. Std confidence = support / body size $= \frac{3}{6} = \frac{1}{2}$. PCA confidence $= \frac{3}{4}$.

## 4. Raise the minimum support

Before running the next cell, predict whether the path rule will appear at minimum support 4. Could any further refinement of this rule reach support 4 by adding positive body atoms while keeping its head fixed? Justify your answer, then check the output.

**Answer** : Support of this rule is 3, so it won't be displayed in the output, since it is below the minimum threshold of 4 for display. And no, no refinement on the body of this rule could reach support 4 : adding constraints to the body makes just the rule more speicific, there's kind of an anti-monotony here that can only shrink predictions: P' ⊆ P, so P' ∩ H ⊆ P ∩ H. The output verifies that.

In [15]:
inspect_amie(4)

Minimum support: 4
REPLAY: no new runtime measurement
Native columns: Rule | Head Coverage | Standard Confidence | Pca Confidence | Support | Body Size | Pca Body Size | Functional Variable
Native rule rows: 0


## 5. Report the comparison

Record the data file, target relation, atom limit, both support thresholds and the mode used (saved output or local run). List the rules returned at each threshold. Use the support reported by AMIE to explain the difference. State one conclusion for this graph and configuration and one limitation of the experiment.

**Answer** : Data file is toy.tsv, target relation is associated_with, atom limit is 3 (head included), thresholds are 1 and 4, and I used the saved output (replay mode), not a local run. With threshold 1, AMIE returns only the path rule ?a interacts_with ?f ?f treats ?b => ?a associated_with ?b, with support 3, std confidence 0.5 and PCA confidence 0.75. With threshold 4, nothing is returned. That's because the support of the rule is 3, so it's above 1 but below 4, and since refinements can't have a bigger support the whole branch is pruned (search space goes from 8 to 3). Conclusion : on this graph the path rule is the only one found and it doesn't survive above a threshold of 3; also PCA confidence is higher than std confidence because g5 and g6 have no recorded disease so they're not counted in the PCA denominator. Limitation : the graph is very small and synthetic (19 facts), so we can't really generalize anything from it.